In [1]:
import os
from dotenv import load_dotenv
load_dotenv()
from langchain_ibm import ChatWatsonx

os.environ["LANGSMITH_API_KEY"]=os.getenv("LANGSMITH_API_KEY")

MODEL_ID = os.getenv('WATSONX_MODEL', "meta-llama/llama-3-3-70b-instruct") #Used for evaluation
WATSONX_APIKEY = os.getenv('WATSONX_APIKEY')
WATSONX_PROJECT_ID = os.getenv('WATSONX_PROJECT_ID')
WATSONX_URL = os.getenv("WATSONX_URL", "https://us-south.ml.cloud.ibm.com")

In [2]:
from langsmith import Client

client = Client()

# Define dataset: these are your test cases
dataset_name = "Chatbot Evaluation"
dataset = client.create_dataset(dataset_name)
client.create_examples(
    dataset_id=dataset.id,
    examples=[
    {
        "inputs": {"question": "Explain what Artificial Intelligence is."},
        "outputs": {"answer": "Artificial Intelligence (AI) is a field of computer science that focuses on building systems capable of performing tasks that normally require human intelligence. These tasks include reasoning, learning from data, understanding language, and making decisions."},
    },
    {
        "inputs": {"question": "What is cloud computing?"},
        "outputs": {"answer": "Cloud computing is the delivery of computing services such as servers, storage, databases, networking, and software over the internet. It allows users to access resources on demand without owning physical infrastructure."},
    },
    {
        "inputs": {"question": "Explain the purpose of Kubernetes."},
        "outputs": {"answer": "Kubernetes is an open-source container orchestration platform used to automate the deployment, scaling, and management of containerized applications. It helps manage large numbers of containers efficiently across clusters of machines."},
    },
    {
        "inputs": {"question": "What is machine learning?"},
        "outputs": {"answer": "Machine learning is a branch of artificial intelligence that enables computers to learn patterns from data and improve their performance without being explicitly programmed. It is widely used in applications such as recommendation systems, fraud detection, and image recognition."},
    },
    {
        "inputs": {"question": "Explain the difference between SQL and NoSQL databases."},
        "outputs": {"answer": "SQL databases use structured schemas and store data in tables with predefined relationships. NoSQL databases are more flexible and can store unstructured or semi-structured data using models like documents, key-value pairs, or graphs."},
    },
    {
        "inputs": {"question": "What are the benefits of using Docker?"},
        "outputs": {"answer": "Docker allows developers to package applications along with their dependencies into containers. This ensures that applications run consistently across different environments such as development, testing, and production."},
    },
    {
        "inputs": {"question": "Explain what an API is."},
        "outputs": {"answer": "An API, or Application Programming Interface, is a set of rules that allows different software systems to communicate with each other. It defines how requests and responses should be structured so applications can exchange data and functionality."},
    },
    {
        "inputs": {"question": "What is the purpose of version control systems?"},
        "outputs": {"answer": "Version control systems help developers track and manage changes to source code over time. They allow teams to collaborate efficiently, maintain a history of modifications, and revert to earlier versions if needed."},
    },
    {
        "inputs": {"question": "Explain what a neural network is."},
        "outputs": {"answer": "A neural network is a machine learning model inspired by the structure of the human brain. It consists of layers of interconnected nodes that process data and learn patterns, making it useful for tasks such as image recognition and natural language processing."},
    },
    {
        "inputs": {"question": "Why is data preprocessing important in machine learning?"},
        "outputs": {"answer": "Data preprocessing prepares raw data for machine learning models by cleaning, transforming, and organizing it. This step improves data quality and helps models learn more accurate patterns, leading to better predictions."},
    },
]
)

{'example_ids': ['e2344972-6b8b-43e3-b327-f737e6613e85',
  'd337b62d-c23e-4a30-a16a-c5c403b58111',
  '284bc01a-f63a-41da-afe1-f1dee5dd6c98',
  '146c3bd1-c7eb-41b8-a41f-9b4b1bf20135',
  'e31ace10-7556-4597-97df-87fbc9357a1a',
  'fff5f21b-5076-4531-b2e3-86d4ccdcc194',
  'de185b70-691d-4802-994e-70572417745f',
  '4e4a0a93-deb3-42e6-8f6d-674906a0558a',
  'e69f4bd8-c5e3-4d52-b2d6-4e4d481ca991',
  '4d9da070-7131-4256-bf2d-8ee0ae853461'],
 'count': 10,
 'as_of': '2026-03-07T13:39:24.792468179Z'}

## Define Metrics (LLM As A Judge)

In [3]:
MODEL_PARAMETERS= {
    "temperature": 0.5,
    "max_tokens": 500,
    "top_p": 0.9,
    "top_k": 50,
    "stop_sequences": ["\n"]
}
 
def create_llm_client(model):
    """Create and return a WatsonX LLM client"""
    llm = ChatWatsonx(
        model_id=model,
        url=WATSONX_URL,
        apikey=WATSONX_APIKEY,
        project_id=WATSONX_PROJECT_ID,
        params=MODEL_PARAMETERS
    )
    return llm


def generate_text(llm, prompt: str):
    """Generate text using the LLM"""
    response = llm.invoke(prompt)
    return response


In [ ]:
from langchain_core.messages import SystemMessage
eval_instructions = "You are an expert professor specialized in grading students' answers to questions."
system_prompt = SystemMessage(content=eval_instructions)

def correctness(inputs:dict,outputs:dict, reference_outputs:dict)->bool:
    user_content = f"""You are grading the following question:
    {inputs['question']}
    Here is the real answer:
    {reference_outputs['answer']}
    You are grading the following predicted answer:
    {outputs['response']}
    Respond with CORRECT or INCORRECT:
    Grade:
    """
    llm = create_llm_client(MODEL_ID)
    system_prompt = SystemMessage(content=eval_instructions
    )
    response=generate_text(llm, prompt= [system_prompt]+[user_content])
    print("Evalutation Output:", response)            

    return response == "CORRECT"

[SystemMessage(content="You are an expert professor specialized in grading students' answers to questions.", additional_kwargs={}, response_metadata={}), 'You are grading the following question:']


In [5]:
## Concisions- checks whether the actual output is less than 2x the length of the expected result.

def concision(outputs: dict, reference_outputs: dict) -> bool:
    return int(len(outputs["response"]) < 2 * len(reference_outputs["answer"]))

## Run Evaluation

In [6]:
default_instructions = "Respond to the users question in a short, concise manner (one short sentence)."
def my_app(question: str,model:str, instructions: str = default_instructions) -> str:
    llm = create_llm_client(model)
    llm_answer = generate_text(llm, prompt= instructions+ '\n'+ question)
    return llm_answer.content


In [7]:
### Call my_app for every datapoints
def ls_target(inputs: str) -> dict:
    return {"response": my_app(inputs["question"],model="meta-llama/llama-3-3-70b-instruct")}

In [8]:
## Run our evaluation
experiment_results=client.evaluate(
    ls_target, ## Your AI system
    data=dataset_name,
    evaluators=[correctness,concision],
    experiment_prefix="llama_70b3.3_wx"
)

View the evaluation results for experiment: 'llama_70b3.3_wx-64fcf732' at:
https://smith.langchain.com/o/5160b42a-75e3-4626-b3e7-d06fe076483e/datasets/dd924be5-6f46-4b47-b41f-ad3f74dd2646/compare?selectedSessions=f692d088-863f-4121-97a6-b53544d32c4a


Evalutation Output: content='Grade: CORRECT' additional_kwargs={} response_metadata={'token_usage': {'completion_tokens': 5, 'prompt_tokens': 159, 'total_tokens': 164}, 'model_name': 'meta-llama/llama-3-3-70b-instruct', 'system_fingerprint': '', 'finish_reason': 'stop'} id='chatcmpl-52437fff1100f2ebf7f7f407838fe397---80fd3c2f-a80e-4500-bec4-1a246ab7c81c' tool_calls=[] invalid_tool_calls=[] usage_metadata={'input_tokens': 159, 'output_tokens': 5, 'total_tokens': 164}
Evalutation Output: content='Grade: INCORRECT' additional_kwargs={} response_metadata={'token_usage': {'completion_tokens': 6, 'prompt_tokens': 141, 'total_tokens': 147}, 'model_name': 'meta-llama/llama-3-3-70b-instruct', 'system_fingerprint': '', 'finish_reason': 'stop'} id='

In [11]:
### Call my_app for every datapoints
def ls_target(inputs: str) -> dict:
    return {"response": my_app(inputs["question"],model="ibm/granite-8b-code-instruct")}
## Run our evaluation
experiment_results=client.evaluate(
    ls_target, ## Your AI system
    data=dataset_name,
    evaluators=[correctness,concision],
    experiment_prefix="granite_8b"
)

View the evaluation results for experiment: 'granite_8b-b246e536' at:
https://smith.langchain.com/o/5160b42a-75e3-4626-b3e7-d06fe076483e/datasets/dd924be5-6f46-4b47-b41f-ad3f74dd2646/compare?selectedSessions=0bf23b84-398d-4f9e-bd53-2cc0d30b563b


Evalutation Output: content='Grade: CORRECT' additional_kwargs={} response_metadata={'token_usage': {'completion_tokens': 5, 'prompt_tokens': 160, 'total_tokens': 165}, 'model_name': 'meta-llama/llama-3-3-70b-instruct', 'system_fingerprint': '', 'finish_reason': 'stop'} id='chatcmpl-879e8bfc99f1b3a7a18c505fe09d6680---b9b31556-0d43-49d5-b031-d6331abaab11' tool_calls=[] invalid_tool_calls=[] usage_metadata={'input_tokens': 160, 'output_tokens': 5, 'total_tokens': 165}
Evalutation Output: content='Grade: CORRECT' additional_kwargs={} response_metadata={'token_usage': {'completion_tokens': 5, 'prompt_tokens': 214, 'total_tokens': 219}, 'model_name': 'meta-llama/llama-3-3-70b-instruct', 'system_fingerprint': '', 'finish_reason': 'stop'} id='chatcmp

### Langsmit dashboard of our expermints with llama and granite 8b


![Langsmith experiments dashboard](./images/langsmith-llm-exps.png)